# Módulo B.3 — Stacking y bandas de idoneidad

Combina los cuatro modelos en un *stacking*, traduce las probabilidades en categorías (No apta/Baja/Media/Alta), evalúa el caso de Turrialba utilizando datos reales y guarda el modelo final.

## 1. Instalar librerías

In [1]:
!pip install xgboost lightgbm rasterio -q

## 2. Montar Drive y cargar el dataset

In [2]:
import os
from pathlib import Path
import pandas as pd

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

CARPETA_BASE = Path('/content/drive/MyDrive/TFM_TeaSuitability')
CARPETA_CLIMA = CARPETA_BASE / 'worldclim'
RUTA_CSV = CARPETA_BASE / 'dataset_master_enriquecido.csv'

if not RUTA_CSV.exists():
    raise FileNotFoundError('Falta el enriquecido. Ejecuta el A.2.')

dataset = pd.read_csv(RUTA_CSV)
print('Cargado. Forma:', dataset.shape)

Mounted at /content/drive
Cargado. Forma: (1333, 14)


## 3. Importaciones

In [3]:
import numpy as np                              # sirve para cálculo numérico
import joblib                                   # sirve para guardar el modelo (.pkl)
import rasterio                                 # sirve para muestrear Turrialba
from rasterio.warp import transform as warp_transform  # sirve para el pH (SoilGrids)
from lightgbm import LGBMClassifier             # sirve para el modelo LightGBM
from sklearn.ensemble import (RandomForestClassifier,   # sirve para Random Forest
                              StackingClassifier)        # sirve para el stacking
from sklearn.linear_model import LogisticRegression     # sirve como meta-modelo
from sklearn.pipeline import make_pipeline      # sirve para encadenar escalado + SVM
from sklearn.preprocessing import StandardScaler # sirve para estandarizar
from sklearn.svm import SVC                       # sirve para el modelo SVM
import warnings; warnings.filterwarnings('ignore') # oculta avisos

## 4. Preparación

In [4]:
# Predictoras = las 8 variables ambientales
columnas_predictoras = [
    'temperatura_media',        # bio1: temperatura media anual (óptimo del té: 18-25 °C)
    'rango_diurno',             # bio2: diferencia entre la máxima del día y la mínima nocturna
    'precipitacion_anual',      # bio12: lluvia total anual (óptimo 1500-3000 mm)
    'estacionalidad_precip',    # bio15: variación de la lluvia entre estaciones
    'precip_trimestre_seco',    # bio17: lluvia en la estación seca (estrés hídrico)
    'elevacion',                # altitud del terreno (el té se cultiva hasta ~2200 m)
    'ph_suelo',                 # pH del suelo (SoilGrids); el té prefiere ácido, 4.5-5.5
]
X = dataset[columnas_predictoras]
y = dataset['clase']

## 5. Funciones de modelado y clasificación

In [5]:
def crear_stacking():
    """
    Crea el stacking con RF + SVM + LightGBM (sin XGBoost).

    Returns
    -------
    sklearn.ensemble.StackingClassifier
        El modelo de stacking sin entrenar.
    """
    base = [
        ('random_forest', RandomForestClassifier(
            n_estimators=300, class_weight='balanced', random_state=42)),
        ('svm', make_pipeline(
            StandardScaler(),
            SVC(probability=True, class_weight='balanced', random_state=42))),
        ('lightgbm', LGBMClassifier(
            n_estimators=300, class_weight='balanced',
            random_state=42, verbose=-1)),
    ]
    return StackingClassifier(
        estimators=base,
        final_estimator=LogisticRegression(max_iter=1000),
        cv=5, stack_method='predict_proba')


def probabilidad_a_banda(probabilidad):
    """Traduce una probabilidad (0-1) a banda de idoneidad."""
    if probabilidad < 0.25:
        return 'No apta'
    if probabilidad < 0.50:
        return 'Baja'
    if probabilidad < 0.75:
        return 'Media'
    return 'Alta'


def muestrear_punto(lon, lat, capas):
    """Extrae las variables climáticas de un punto desde los rasters (WorldClim)."""
    valores = {}
    for nombre, ruta in capas.items():
        with rasterio.open(ruta) as raster:
            valores[nombre] = float(next(raster.sample([(lon, lat)]))[0])
    return pd.DataFrame([valores])


# URL del mapa de pH de SoilGrids (media, 0-5 cm), lectura remota
URL_PH = ('/vsicurl/https://files.isric.org/soilgrids/latest/data/'
          'phh2o/phh2o_0-5cm_mean.vrt')


def muestrear_ph_soilgrids(lon, lat, url=URL_PH):
    """
    Obtiene el pH del suelo (SoilGrids) en un punto (lon, lat).

    Transforma la coordenada a la proyección Homolosine de SoilGrids y
    muestrea el pH (los valores vienen x10, se dividen entre 10). Devuelve
    NaN si no hay dato en ese punto.
    """
    with rasterio.open(url) as src:
        xs, ys = warp_transform('EPSG:4326', src.crs, [lon], [lat])
        v = float(next(src.sample(list(zip(xs, ys))))[0])
    if v == src.nodata:
        return np.nan
    return v / 10.0


def valorar_ph(ph):
    """
    Interpreta el pH del suelo para el cultivo de té.

    El té prospera en suelos ácidos (pH óptimo ~4.5-5.5).
    """
    if ph <= 5.5:
        return 'ácido, IDEAL para el té'
    if ph <= 6.5:
        return 'ligeramente ácido, aceptable'
    return 'poco ácido/alcalino, MENOS favorable para el té'

## 6. Entrenar el stacking con todos los datos reales

In [6]:
modelo = crear_stacking()
modelo.fit(X, y)
print('Stacking (RF+SVM+LightGBM) entrenado con', len(X), 'puntos y',
      X.shape[1], 'variables (incluye pH del suelo).')

Stacking (RF+SVM+LightGBM) entrenado con 1333 puntos y 7 variables (incluye pH del suelo).


## 7. Predecir la idoneidad de todos los puntos

In [7]:
probabilidades = modelo.predict_proba(X)[:, 1]
dataset['prob_idoneidad'] = probabilidades
dataset['banda_idoneidad'] = [probabilidad_a_banda(p) for p in probabilidades]
print('Distribución de bandas:')
print(dataset['banda_idoneidad'].value_counts())

Distribución de bandas:
banda_idoneidad
Alta       766
No apta    529
Baja        25
Media       13
Name: count, dtype: int64


## 8. Caso Turrialba (con sus condiciones reales, incluido el pH)

Se toman muestras de las variables climáticas de los rasters y del pH del suelo de SoilGrids. En caso de que el pH no esté disponible en un punto específico, se reemplaza con la mediana del conjunto, utilizando la misma estrategia que se aplica para los datos faltantes en el dataset.

In [8]:
# 1- variables climáticas de Turrialba desde los rasters de WorldClim
CAPAS = {
    'temperatura_media':      CARPETA_CLIMA / 'wc2.1_5m_bio_1.tif',   # bio1
    'rango_diurno':           CARPETA_CLIMA / 'wc2.1_5m_bio_2.tif',   # bio2
    'precipitacion_anual':    CARPETA_CLIMA / 'wc2.1_5m_bio_12.tif',  # bio12
    'estacionalidad_precip':  CARPETA_CLIMA / 'wc2.1_5m_bio_15.tif',  # bio15
    'precip_trimestre_seco':  CARPETA_CLIMA / 'wc2.1_5m_bio_17.tif',  # bio17
    'elevacion':              CARPETA_CLIMA / 'wc2.1_5m_elev.tif',    # elevación
}
turrialba = muestrear_punto(-83.6855, 9.9044, CAPAS)

# 2- pH del suelo de Turrialba (SoilGrids); si falla o sale NaN, mediana del dataset
try:
    ph_turrialba = muestrear_ph_soilgrids(-83.6855, 9.9044)
except Exception:
    ph_turrialba = np.nan
if pd.isna(ph_turrialba):
    ph_turrialba = dataset['ph_suelo'].median()
    print('(pH no disponible en el punto; se usa la mediana del dataset)')
turrialba['ph_suelo'] = ph_turrialba

# 3- ordenar columnas igual que en el entrenamiento y predecir
turrialba = turrialba[columnas_predictoras]
prob_t = modelo.predict_proba(turrialba)[0, 1]

print('Condiciones reales de Turrialba:')
print(turrialba.round(1).to_dict('records')[0])
print(f"\npH del suelo: {ph_turrialba:.1f} -> {valorar_ph(ph_turrialba)}")
print(f'Probabilidad de idoneidad: {prob_t:.1%}')
print(f'Banda: {probabilidad_a_banda(prob_t)}')

(pH no disponible en el punto; se usa la mediana del dataset)
Condiciones reales de Turrialba:
{'temperatura_media': 21.6, 'rango_diurno': 9.8, 'precipitacion_anual': 2955.0, 'estacionalidad_precip': 33.7, 'precip_trimestre_seco': 367.0, 'elevacion': 897.0, 'ph_suelo': 5.5}

pH del suelo: 5.5 -> ácido, IDEAL para el té
Probabilidad de idoneidad: 58.5%
Banda: Media


## 9. Guardar el modelo y las predicciones

In [9]:
ruta_modelo = CARPETA_BASE / 'modelo_teasuitability_real.pkl'
joblib.dump(modelo, ruta_modelo)   # guarda el modelo final (8 variables)
dataset.to_csv(CARPETA_BASE / 'dataset_real_con_predicciones.csv', index=False)
print('Modelo guardado en:', ruta_modelo)

Modelo guardado en: /content/drive/MyDrive/TFM_TeaSuitability/modelo_teasuitability_real.pkl
